# Algorithm Discovery via ZX Motif Phylogeny

Data-driven pipeline that uses structural insights from motif phylogeny
analysis to construct novel quantum circuits. Four discovery strategies:

1. **Coverage Gap Targeting** — fills under-represented families
2. **Cross-Family Bridge** — hybridises high-similarity cross-family pairs
3. **Irreducible Composition** — composes motifs surviving full ZX reduction
4. **PCA Void Filling** — targets empty regions in fingerprint PCA space

In [ ]:
import copy
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyzx as zx
from matplotlib.patches import Patch
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

from zx_motifs.algorithms.registry import ALGORITHM_FAMILY_MAP, REGISTRY
from zx_motifs.pipeline.converter import SimplificationLevel, convert_at_all_levels, qiskit_to_zx
from zx_motifs.pipeline.featurizer import pyzx_to_networkx
from zx_motifs.pipeline.matcher import MotifPattern, find_motif_in_graph
from zx_motifs.pipeline.fingerprint import build_corpus, discover_motifs, build_fingerprint_matrix

%matplotlib inline

## 1. Load Phylogeny Results

Build corpus and fingerprint matrix (or load from notebook 02 outputs).

In [ ]:
corpus = build_corpus(max_qubits=5)
motifs = discover_motifs(corpus)
counts_df, freq_df = build_fingerprint_matrix(corpus, motifs)
print(f"Corpus: {len({n for n,_ in corpus})} instances, {len(motifs)} motifs")
print(f"Fingerprint: {counts_df.shape}")

## 2. Helpers & Data Types

In [ ]:
FAMILY_COLOURS = {
    "oracle": "#e41a1c", "entanglement": "#377eb8", "error_correction": "#4daf4a",
    "distillation": "#984ea3", "protocol": "#ff7f00", "variational": "#a65628",
    "simulation": "#f781bf", "transform": "#999999", "arithmetic": "#66c2a5",
    "machine_learning": "#e6ab02", "linear_algebra": "#1b9e77", "cryptography": "#d95f02",
    "sampling": "#7570b3", "error_mitigation": "#e7298a", "topological": "#66a61e",
    "metrology": "#e6ab02", "differential_equations": "#a6761d", "tda": "#666666",
    "communication": "#1f78b4",
}

STRATEGY_COLOURS = {
    "coverage_gap": "#e41a1c", "cross_family": "#377eb8",
    "irreducible": "#4daf4a", "pca_void": "#984ea3",
}

def _get_family(instance_name):
    base = instance_name.rsplit("_q", 1)[0]
    return ALGORITHM_FAMILY_MAP.get(base, "unknown")

@dataclass
class CandidateSpec:
    name: str
    strategy: str
    motif_ids: list[str] = field(default_factory=list)
    rationale: str = ""
    n_qubits: int = 4
    source_algo_a: str | None = None
    source_algo_b: str | None = None
    shared_motifs: list[str] = field(default_factory=list)

## 3. Motif Template Registry

Map motif IDs to gate-template functions that append gates to a QuantumCircuit at a given qubit offset.

In [ ]:
def _tpl_phase_gadget_2t(qc, off):
    qc.cx(off, off + 1); qc.t(off + 1); qc.cx(off, off + 1)
    return 2

def _tpl_phase_gadget_3t(qc, off):
    qc.cx(off, off + 1); qc.cx(off, off + 2); qc.t(off + 2)
    qc.cx(off, off + 2); qc.cx(off, off + 1)
    return 3

def _tpl_cx_pair(qc, off):
    qc.cx(off, off + 1)
    return 2

def _tpl_hadamard_sandwich(qc, off):
    qc.h(off); qc.s(off); qc.h(off)
    return 1

def _tpl_zz_interaction(qc, off):
    qc.cx(off, off + 1); qc.rz(0.7, off + 1); qc.cx(off, off + 1)
    return 2

def _tpl_syndrome_extraction(qc, off):
    qc.cx(off, off + 1); qc.cx(off, off + 2)
    return 3

def _tpl_toffoli_core(qc, off):
    qc.t(off); qc.cx(off, off + 1); qc.t(off + 1)
    return 2

def _tpl_cluster_chain(qc, off):
    qc.h(off); qc.cz(off, off + 1); qc.h(off + 1)
    return 2

def _tpl_trotter_layer(qc, off):
    qc.cx(off, off + 1); qc.rz(0.5, off + 1)
    qc.cx(off + 1, off + 2); qc.rz(0.3, off + 2)
    return 3

def _tpl_x_hub_3z(qc, off):
    qc.cx(off + 1, off); qc.cx(off + 2, off); qc.cx(off + 3, off)
    return 4

TEMPLATES = {
    "phase_gadget_2t": (2, _tpl_phase_gadget_2t),
    "phase_gadget_3t": (3, _tpl_phase_gadget_3t),
    "cx_pair": (2, _tpl_cx_pair),
    "hadamard_sandwich": (1, _tpl_hadamard_sandwich),
    "zz_interaction": (2, _tpl_zz_interaction),
    "syndrome_extraction": (3, _tpl_syndrome_extraction),
    "toffoli_core": (2, _tpl_toffoli_core),
    "cluster_chain": (2, _tpl_cluster_chain),
    "trotter_layer": (3, _tpl_trotter_layer),
    "x_hub_3z_param": (4, _tpl_x_hub_3z),
}
print(f"{len(TEMPLATES)} motif templates registered")

## 4. Cross-Level Survival

Identify motifs that survive `full_reduce` — these are irreducible and will be key inputs to Strategy 3.

In [ ]:
from zx_motifs.pipeline.converter import SimplificationLevel

levels = [lvl.value for lvl in SimplificationLevel]
seen_bases = {}
for name, level in corpus:
    base = name.rsplit("_q", 1)[0]
    if base not in seen_bases or name < seen_bases[base]:
        seen_bases[base] = name
representatives = sorted(set(seen_bases.values()))

motif_ids = [mp.motif_id for mp in motifs]
survival = np.zeros((len(motifs), len(levels)))
for j, level in enumerate(levels):
    for rep in representatives:
        key = (rep, level)
        if key not in corpus:
            continue
        host = corpus[key]
        for i, mp in enumerate(motifs):
            if find_motif_in_graph(mp.graph, host, max_matches=1):
                survival[i, j] += 1
if representatives:
    survival /= len(representatives)

fr_idx = levels.index("full_reduce") if "full_reduce" in levels else -1
survivors = [motif_ids[i] for i in range(len(motif_ids))
             if fr_idx >= 0 and survival[i, fr_idx] > 0]
print(f"Irreducible motifs (survive full_reduce): {survivors}")
available_survivors = [s for s in survivors if s in TEMPLATES]
print(f"With templates: {available_survivors}")

## 5. Universality & Coverage Analysis

In [ ]:
family_of = {inst: _get_family(inst) for inst in counts_df.index}
all_families = sorted(set(family_of.values()))
n_families = len(all_families)

motif_families = {}
motif_category = {}
for mid in counts_df.columns:
    col = counts_df[mid]
    present = col[col > 0].index.tolist()
    fams = set(family_of[inst] for inst in present)
    motif_families[mid] = fams
    n_fam = len(fams)
    if n_fam >= n_families * 0.8:
        motif_category[mid] = "universal"
    elif n_fam >= n_families * 0.4:
        motif_category[mid] = "common"
    elif n_fam > 0:
        motif_category[mid] = "specific"
    else:
        motif_category[mid] = "unused"

# Coverage
from zx_motifs.pipeline.decomposer import decompose_across_corpus
dec_results = decompose_across_corpus(corpus, motifs, target_level="spider_fused")
coverage_by_family = {}
for algo_name, dr in dec_results.items():
    base = algo_name.rsplit("_q", 1)[0]
    fam = ALGORITHM_FAMILY_MAP.get(base, "unknown")
    coverage_by_family.setdefault(fam, []).append(dr.coverage_ratio)
avg_coverage = {f: np.mean(v) for f, v in coverage_by_family.items()}
print("Coverage by family:")
for f, c in sorted(avg_coverage.items(), key=lambda x: x[1]):
    print(f"  {f:20s}: {c:.1%}")

## 6. Discovery Strategies

### Strategy 3: Irreducible Composition (primary)

Compose pairs of motifs that survive full ZX reduction. This is the strategy that discovered **irr_pair11**.

In [ ]:
candidates = []
seen_pairs = set()
pair_counter = 0

for i, m1 in enumerate(available_survivors):
    for m2 in available_survivors[i + 1:]:
        total_q = TEMPLATES[m1][0] + TEMPLATES[m2][0]
        if total_q > 6:
            continue
        pair_key = tuple(sorted([m1, m2]))
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)
        pair_counter += 1
        candidates.append(CandidateSpec(
            name=f"irr_pair{pair_counter:02d}",
            strategy="irreducible",
            motif_ids=[m1, m2],
            rationale=f"Irreducible pair: {m1} + {m2} (both survive full ZX reduction)",
            n_qubits=min(6, total_q + 1),
        ))

print(f"Strategy 3 (irreducible): {len(candidates)} candidates")
for c in candidates[:5]:
    print(f"  {c.name}: {c.motif_ids} ({c.n_qubits}q)")
if len(candidates) > 5:
    print(f"  ... and {len(candidates) - 5} more")

## 7. Build Circuits from Specs

In [ ]:
def build_circuit(spec):
    qc = QuantumCircuit(spec.n_qubits)
    offset = 0
    applied = 0
    for mid in spec.motif_ids:
        if mid not in TEMPLATES:
            continue
        tpl_qubits, apply_fn = TEMPLATES[mid]
        if offset + tpl_qubits > spec.n_qubits:
            offset = 0
        try:
            apply_fn(qc, offset)
            applied += 1
        except Exception:
            continue
        offset = (offset + 1) % max(1, spec.n_qubits - tpl_qubits + 1)
    return qc if applied > 0 else None

built = []
for spec in candidates:
    qc = build_circuit(spec)
    if qc is not None:
        built.append((spec, qc))
        
print(f"Successfully built {len(built)}/{len(candidates)} circuits")
for spec, qc in built[:3]:
    print(f"  {spec.name}: {qc.num_qubits}q, {qc.size()} gates, depth={qc.depth()}")

## 8. Validate Candidates

Check unitarity and ZX tensor preservation.

In [ ]:
validations = {}
for spec, qc in built:
    op = Operator(qc)
    mat = op.data
    n = mat.shape[0]
    product = mat @ mat.conj().T
    identity_err = float(np.linalg.norm(product - np.eye(n)) / n)
    is_unitary = identity_err < 1e-6
    
    zx_circ = qiskit_to_zx(qc)
    g = zx_circ.to_graph()
    g_copy = copy.deepcopy(g)
    zx.simplify.full_reduce(g_copy)
    try:
        tensors_match = zx.compare_tensors(g, g_copy)
    except Exception:
        tensors_match = None
    
    validations[spec.name] = {
        "is_unitary": is_unitary,
        "tensors_match": tensors_match,
        "n_gates": qc.size(),
        "depth": qc.depth(),
        "zx_vertices_raw": g.num_vertices(),
        "zx_vertices_reduced": g_copy.num_vertices(),
    }
    status = "PASS" if is_unitary else "FAIL"
    print(f"  {spec.name}: {status} (unitary_err={identity_err:.1e}, "
          f"ZX {g.num_vertices()}\u2192{g_copy.num_vertices()} vertices)")

## 9. Fingerprint & Score Candidates

Place candidates in the existing fingerprint space and compute novelty scores.

In [ ]:
# Fingerprint candidates
for spec, qc in built:
    instance = f"{spec.name}_q{qc.num_qubits}"
    try:
        snapshots = convert_at_all_levels(qc, instance)
        for snap in snapshots:
            nxg = pyzx_to_networkx(snap.graph, coarsen_phases=True)
            corpus[(instance, snap.level.value)] = nxg
    except Exception as e:
        print(f"  Warning: {instance}: {e}")

cand_instances = [f"{spec.name}_q{qc.num_qubits}" for spec, qc in built]
cand_counts = np.zeros((len(cand_instances), len(motifs)), dtype=int)
for i, inst in enumerate(cand_instances):
    key = (inst, "spider_fused")
    if key not in corpus:
        continue
    host = corpus[key]
    for j, mp in enumerate(motifs):
        matches = find_motif_in_graph(mp.graph, host, max_matches=100)
        cand_counts[i, j] = len(matches)

motif_id_list = [mp.motif_id for mp in motifs]
cand_counts_df = pd.DataFrame(cand_counts, index=cand_instances, columns=motif_id_list)
row_sums = cand_counts_df.sum(axis=1).replace(0, 1)
cand_freq_df = cand_counts_df.div(row_sums, axis=0)

# Score: cosine novelty
existing_mat = freq_df.fillna(0).values
scored = {}
for ci, cname in enumerate(cand_instances):
    cv = cand_freq_df.iloc[ci].reindex(freq_df.columns, fill_value=0).values
    cv_norm = np.linalg.norm(cv)
    sims = []
    for ei in range(existing_mat.shape[0]):
        ev = existing_mat[ei]
        ev_norm = np.linalg.norm(ev)
        if cv_norm < 1e-10 or ev_norm < 1e-10:
            sims.append(0.0)
        else:
            sims.append(float(np.dot(cv, ev) / (cv_norm * ev_norm)))
    best_idx = int(np.argmax(sims))
    nearest = list(freq_df.index)[best_idx]
    scored[cname] = {
        "nearest": nearest,
        "nearest_family": _get_family(nearest),
        "similarity": round(max(sims), 4),
        "novelty": round(1 - max(sims), 4),
    }

scored_df = pd.DataFrame(scored).T.sort_values("novelty", ascending=False)
print("Top candidates by novelty:")
print(scored_df.head(10).to_string())

## 10. PCA Visualization

Overlay candidates on the existing algorithm PCA scatter.

In [ ]:
# PCA of existing
mat = freq_df.fillna(0).values
mean = mat.mean(axis=0)
mat_c = mat - mean
U, S, Vt = np.linalg.svd(mat_c, full_matrices=False)
coords = U[:, :2] * S[:2]
var_explained = (S[:2] ** 2) / max((S**2).sum(), 1e-10)

fig, ax = plt.subplots(figsize=(12, 9))
instances = list(freq_df.index)
families = [_get_family(inst) for inst in instances]

for fam in sorted(set(families)):
    idx = [i for i, f in enumerate(families) if f == fam]
    ax.scatter(coords[idx, 0], coords[idx, 1],
               c=FAMILY_COLOURS.get(fam, "#333"), label=fam, alpha=0.35, s=25)

# Project candidates
for ci, cname in enumerate(cand_instances):
    cv = cand_freq_df.iloc[ci].reindex(freq_df.columns, fill_value=0).values
    cand_pca = (cv - mean) @ Vt[:2].T
    label_short = cname.rsplit("_q", 1)[0][:15]
    ax.scatter(cand_pca[0], cand_pca[1], marker="p", s=250,
               c=STRATEGY_COLOURS["irreducible"], edgecolors="black", linewidths=1, zorder=5)
    ax.annotate(label_short, (cand_pca[0], cand_pca[1]),
                textcoords="offset points", xytext=(8, 8), fontsize=7, fontweight="bold")

ax.set_xlabel(f"PC1 ({var_explained[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({var_explained[1]:.1%} variance)")
ax.set_title("Algorithm Phylogeny \u2014 Discovery Candidates Overlaid")
handles, labels = ax.get_legend_handles_labels()
seen = set()
unique_h, unique_l = [], []
for h, l in zip(handles, labels):
    if l not in seen:
        seen.add(l)
        unique_h.append(h)
        unique_l.append(l)
ax.legend(unique_h, unique_l, fontsize=6, ncol=2, loc="upper left")
plt.tight_layout()
plt.show()

## Key Finding

**irr_pair11** emerges from Strategy 3 (irreducible composition) by combining
`phase_gadget_3t` + `cluster_chain` — two motifs that survive full ZX reduction.
This circuit shows unique structural properties that make it a promising VQE ansatz
entangler, evaluated in detail in notebook 04.